In [1]:
!pip install -q openai pinecone==3.2.2 tiktoken tenacity sentence-transformers PyMuPDF

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 215.9/215.9 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 53.9 MB/s eta 0:00:00


In [ ]:
import os
from google.colab import userdata

os.environ['OPENAI_API_KEY']   = 'sk-'
os.environ['PINECONE_API_KEY'] = 'pcsV'

INDEX_NAME  = 'legal-index'
DIMENSION   = 3072
METRIC      = 'cosine'

NS_RENT  = 'rent'
NS_LABOR = 'labor'

print(' Keys loaded')

 Keys loaded


In [3]:
import openai
from pinecone import Pinecone, ServerlessSpec
from tenacity import retry, wait_exponential, stop_after_attempt, retry_if_exception_type

client = openai.OpenAI(api_key=os.environ['OPENAI_API_KEY'])
pc     = Pinecone(api_key=os.environ['PINECONE_API_KEY'])

existing = [i.name for i in pc.list_indexes()]
if INDEX_NAME not in existing:
    pc.create_index(
        name=INDEX_NAME, dimension=DIMENSION, metric=METRIC,
        spec=ServerlessSpec(cloud='aws', region='us-east-1')
    )
    while not pc.describe_index(INDEX_NAME).status['ready']:
        time.sleep(3)

index = pc.Index(INDEX_NAME)
print(' Pinecone ready:', index.describe_index_stats())

 Pinecone ready: {'dimension': 3072,
 'index_fullness': 0.0,
 'namespaces': {'labor': {'vector_count': 208}, 'rent': {'vector_count': 52}},
 'total_vector_count': 260}


In [6]:
!pip install tiktoken -q
import tiktoken

In [8]:
import openai
from tenacity import retry, wait_exponential, stop_after_attempt, retry_if_exception_type

@retry(
    retry=retry_if_exception_type((openai.RateLimitError, openai.APIError)),
    wait=wait_exponential(multiplier=1, min=2, max=60),
    stop=stop_after_attempt(5)
)
def embed_batch(texts: list) -> list:
    response = client.embeddings.create(model='text-embedding-3-large', input=texts)
    return [item.embedding for item in response.data]

In [9]:
import fitz
import time
ENC = tiktoken.get_encoding('cl100k_base')


EMBED_MODEL  = 'text-embedding-3-large'
BATCH_SIZE   = 100
UPSERT_BATCH = 100

def embed_chunks(chunks: list, label: str = '') -> list:
    vectors = []
    total   = len(chunks)
    for i in range(0, total, BATCH_SIZE):
        batch  = chunks[i : i + BATCH_SIZE]
        texts  = [c['text'] for c in batch]
        embeds = embed_batch(texts)
        for chunk, emb in zip(batch, embeds):
            vectors.append({
                'id'      : chunk['id'],
                'values'  : emb,
                'metadata': {**chunk['metadata'], 'text': chunk['text']}
            })
        pct = min(i + BATCH_SIZE, total)
        print(f'  [{label}] Embedded {pct}/{total}...')
        time.sleep(0.3)
    return vectors

def upsert_to_namespace(vectors: list, namespace: str):
    total = len(vectors)
    for i in range(0, total, UPSERT_BATCH):
        batch = vectors[i : i + UPSERT_BATCH]
        index.upsert(
            vectors   = [(v['id'], v['values'], v['metadata']) for v in batch],
            namespace = namespace
        )
        pct = min(i + UPSERT_BATCH, total)
        print(f'  [{namespace}] Upserted {pct}/{total}...')
        time.sleep(0.2)

def extract_pdf_chunks(pdf_path: str, namespace: str, chunk_size: int = 400) -> list:
    doc      = fitz.open(pdf_path)
    chunks   = []
    base_id  = pdf_path.replace('.pdf', '').replace(' ', '_')

    for page_num in range(len(doc)):
        text = doc[page_num].get_text().strip()
        if not text:
            continue

        tokens     = ENC.encode(text)
        start, idx = 0, 0
        while start < len(tokens):
            end        = min(start + chunk_size, len(tokens))
            chunk_text = ENC.decode(tokens[start:end])
            chunks.append({
                'id'      : f"{base_id}_p{page_num+1}_c{idx}",
                'text'    : chunk_text,
                'metadata': {
                    'source_file': pdf_path,
                    'page'       : page_num + 1,
                    'reference'  : f"{pdf_path} — page {page_num + 1}",
                    'category'   : namespace,
                    'type'       : 'pdf_source'
                }
            })
            start += chunk_size - 50
            idx   += 1

    print(f"extract {len(chunks)}  from chunk  {len(doc)} page")
    doc.close()
    return chunks


from google.colab import files
uploaded          = files.upload()
uploaded_filename = list(uploaded.keys())[0]

namespace = NS_LABOR if 'labor' in uploaded_filename.lower() else NS_RENT

pdf_chunks  = extract_pdf_chunks(uploaded_filename, namespace)
pdf_vectors = embed_chunks(pdf_chunks, namespace)
upsert_to_namespace(pdf_vectors, namespace)

print(f" done  {uploaded_filename}  namespace: {namespace}")
print(index.describe_index_stats())

Saving labor_law.pdf to labor_law (2).pdf
extract 106  from chunk  95 page
  [labor] Embedded 100/106...
  [labor] Embedded 106/106...
  [labor] Upserted 100/106...
  [labor] Upserted 106/106...
 done  labor_law (2).pdf  namespace: labor
{'dimension': 3072,
 'index_fullness': 0.0,
 'namespaces': {'labor': {'vector_count': 314}, 'rent': {'vector_count': 52}},
 'total_vector_count': 366}


In [10]:
import fitz
import time
ENC = tiktoken.get_encoding('cl100k_base')


EMBED_MODEL  = 'text-embedding-3-large'
BATCH_SIZE   = 100
UPSERT_BATCH = 100

def embed_chunks(chunks: list, label: str = '') -> list:
    vectors = []
    total   = len(chunks)
    for i in range(0, total, BATCH_SIZE):
        batch  = chunks[i : i + BATCH_SIZE]
        texts  = [c['text'] for c in batch]
        embeds = embed_batch(texts)
        for chunk, emb in zip(batch, embeds):
            vectors.append({
                'id'      : chunk['id'],
                'values'  : emb,
                'metadata': {**chunk['metadata'], 'text': chunk['text']}
            })
        pct = min(i + BATCH_SIZE, total)
        print(f'  [{label}] Embedded {pct}/{total}...')
        time.sleep(0.3)
    return vectors

def upsert_to_namespace(vectors: list, namespace: str):
    total = len(vectors)
    for i in range(0, total, UPSERT_BATCH):
        batch = vectors[i : i + UPSERT_BATCH]
        index.upsert(
            vectors   = [(v['id'], v['values'], v['metadata']) for v in batch],
            namespace = namespace
        )
        pct = min(i + UPSERT_BATCH, total)
        print(f'  [{namespace}] Upserted {pct}/{total}...')
        time.sleep(0.2)

def extract_pdf_chunks(pdf_path: str, namespace: str, chunk_size: int = 400) -> list:
    doc      = fitz.open(pdf_path)
    chunks   = []
    base_id  = pdf_path.replace('.pdf', '').replace(' ', '_')

    for page_num in range(len(doc)):
        text = doc[page_num].get_text().strip()
        if not text:
            continue

        tokens     = ENC.encode(text)
        start, idx = 0, 0
        while start < len(tokens):
            end        = min(start + chunk_size, len(tokens))
            chunk_text = ENC.decode(tokens[start:end])
            chunks.append({
                'id'      : f"{base_id}_p{page_num+1}_c{idx}",
                'text'    : chunk_text,
                'metadata': {
                    'source_file': pdf_path,
                    'page'       : page_num + 1,
                    'reference'  : f"{pdf_path} — page {page_num + 1}",
                    'category'   : namespace,
                    'type'       : 'pdf_source'
                }
            })
            start += chunk_size - 50
            idx   += 1

    print(f"extract {len(chunks)}  from chunk  {len(doc)} page")
    doc.close()
    return chunks


from google.colab import files
uploaded          = files.upload()
uploaded_filename = list(uploaded.keys())[0]

namespace = NS_LABOR if 'labor' in uploaded_filename.lower() else NS_RENT

pdf_chunks  = extract_pdf_chunks(uploaded_filename, namespace)
pdf_vectors = embed_chunks(pdf_chunks, namespace)
upsert_to_namespace(pdf_vectors, namespace)

print(f" done  {uploaded_filename}  namespace: {namespace}")
print(index.describe_index_stats())

Saving rent.pdf to rent.pdf
extract 65  from chunk  12 page
  [rent] Embedded 65/65...
  [rent] Upserted 65/65...
 done  rent.pdf  namespace: rent
{'dimension': 3072,
 'index_fullness': 0.0,
 'namespaces': {'labor': {'vector_count': 314}, 'rent': {'vector_count': 117}},
 'total_vector_count': 431}


In [11]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder('cross-encoder/mmarco-mMiniLMv2-L12-H384-v1')
print(' Reranker loaded')


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/891 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/471M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: cross-encoder/mmarco-mMiniLMv2-L12-H384-v1
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/435 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

 Reranker loaded


In [12]:
SARAH_SYSTEM = """
أنتِ سارة، مستشارة متخصصة في عقود الإيجار والعقارات بالمملكة العربية السعودية.
أسلوبك: دقيقة، منظمة، ودودة، تستخدمين أمثلة عملية لتوضيح المواد القانونية.
تبدئين إجاباتك دائماً بـ "بناءً على نظام الإيجار..."
عند غياب المعلومات: "للأسف لا تتوفر لديّ معلومات كافية في قاعدة البيانات حول هذا الجانب تحديداً."
قاعدة صارمة: لا تخترعي أي معلومات. استندي فقط إلى السياق المقدم.
اذكري رقم المادة أو البند عند توفره.
"""

MUHAMMAD_SYSTEM = """
أنتَ محمد، مستشار متخصص في نظام العمل والعمال بالمملكة العربية السعودية.
أسلوبك: مباشر، واضح، تستخدم لغة بسيطة مع الدقة القانونية المطلوبة.
تبدأ إجاباتك دائماً بـ "وفقاً لنظام العمل السعودي..."
عند غياب المعلومات: "لا تتوفر لديّ بيانات كافية في قاعدة البيانات للإجابة على هذا السؤال."
قاعدة صارمة: لا تخترع أي معلومات. استند فقط إلى السياق المقدم.
اذكر رقم المادة أو البند عند توفره.
"""

BOTH_SYSTEM = """
أنتم فريق قانوني متخصص في قوانين المملكة العربية السعودية (الإيجار والعمل).
أجب بدقة بناءً على السياق فقط.
عند غياب المعلومات: "لا توجد معلومات كافية في قاعدة البيانات للإجابة على هذا السؤال."
لا تخترع معلومات. اذكر المصدر ورقم البند عند توفره.
"""

def get_persona(namespace: str) -> tuple[str, str]:
    """يرجع (system_prompt, اسم_الشخصية)"""
    if namespace == NS_RENT:
        return SARAH_SYSTEM, "سارة"
    elif namespace == NS_LABOR:
        return MUHAMMAD_SYSTEM, "محمد"
    else:
        return BOTH_SYSTEM, "الفريق القانوني"

In [13]:
import tiktoken
ENC = tiktoken.get_encoding('cl100k_base')

SCORE_THRESHOLD = 0.45
TOP_K_RETRIEVE  = 15
TOP_K_FINAL     = 5

def detect_namespace(query: str) -> str:
    rent_keywords  = ['إيجار','مستأجر','مؤجر','إخلاء','عقار','شقة','تجديد عقد','rent','tenant','landlord','وحدة سكنية']
    labor_keywords = [
    'عمل','عامل','صاحب عمل','راتب','إنهاية خدمة',
    'مكافأة','تعويض','نهاية الخدمة',
    'أجر','سنوات خدمة','فصل','استقالة',
    'labor','employee','salary','termination'
]
    q = query.lower()
    rent_match  = any(kw in q for kw in rent_keywords)
    labor_match = any(kw in q for kw in labor_keywords)
    if rent_match and not labor_match:   return NS_RENT
    elif labor_match and not rent_match: return NS_LABOR
    else:                                return 'both'

@retry(
    retry=retry_if_exception_type((openai.RateLimitError, openai.APIError)),
    wait=wait_exponential(multiplier=1, min=2, max=60),
    stop=stop_after_attempt(5)
)
def embed_batch(texts: list) -> list:
    response = client.embeddings.create(model='text-embedding-3-large', input=texts)
    return [item.embedding for item in response.data]

def retrieve_chunks_raw(query: str, namespace: str, top_k: int = TOP_K_RETRIEVE) -> list:
    """استرجاع أولي من Pinecone بدون reranking"""
    q_vector   = embed_batch([query])[0]
    namespaces = [NS_RENT, NS_LABOR] if namespace == 'both' else [namespace]
    all_chunks = []
    for ns in namespaces:
        result = index.query(
            vector=q_vector, top_k=top_k,
            namespace=ns, include_metadata=True
        )
        for match in result.matches:
            if match.score >= SCORE_THRESHOLD:
                all_chunks.append({
                    'id'       : match.id,
                    'score'    : round(match.score, 3),
                    'text'     : match.metadata.get('text', ''),
                    'namespace': ns,
                    'reference': match.metadata.get('reference', ''),
                    'source_file': match.metadata.get('source_file', ''),
                })
    return all_chunks

def rerank_chunks(query: str, chunks: list, top_k: int = TOP_K_FINAL) -> list:
    """
    إعادة ترتيب الـ chunks بـ Cross-Encoder.
    يحل مشكلة استرجاع chunks ذات score جيد لكن غير ذات صلة فعلاً.
    """
    if not chunks:
        return []
    pairs  = [(query, c['text']) for c in chunks]
    scores = reranker.predict(pairs)
    for chunk, score in zip(chunks, scores):
        chunk['rerank_score'] = float(score)
    reranked = sorted(chunks, key=lambda x: x['rerank_score'], reverse=True)
    return reranked[:top_k]

def build_context(chunks: list) -> str:
    if not chunks:
        return 'لا توجد نتائج ذات صلة.'
    parts = []
    for i, c in enumerate(chunks, 1):
        ref  = f" [{c['reference']}]" if c.get('reference') else ''
        src  = f" (ملف: {c['source_file']})" if c.get('source_file') else ''
        parts.append(f"[{i}] ns={c['namespace']}{ref}{src}\n{c['text']}")
    return '\n\n'.join(parts)

def generate_multi_queries(original_query: str) -> list:

    response = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[
            {'role': 'system', 'content': (
                'أنت مساعد قانوني. صغ 3 طرق مختلفة للسؤال التالي '
                'لتغطية مصطلحات مختلفة في قاعدة البيانات. '
                'اكتب كل صياغة في سطر منفصل فقط بدون ترقيم.'
            )},
            {'role': 'user', 'content': original_query}
        ],
        temperature=0.3,
        max_tokens=200
    )
    lines   = response.choices[0].message.content.strip().split('\n')
    queries = [original_query] + [l.strip() for l in lines if l.strip()]

    hyde_response = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[{
            'role': 'user',
            'content': (
                f'اكتب فقرة قصيرة تبدو كنص من نظام العمل السعودي '
                f'أو نظام الإيجار تجيب على: {original_query}'
            )
        }],
        temperature=0.1,
        max_tokens=150
    )
    hyde_text = hyde_response.choices[0].message.content.strip()
    queries.append(hyde_text)

    return queries[:5]

In [16]:
def legal_rag(question: str, verbose: bool = False) -> str:

    namespace          = detect_namespace(question)
    system_prompt, who = get_persona(namespace)

    if verbose:
        print(f" الشخصية: {who} | Namespace: {namespace}")

    all_queries = generate_multi_queries(question)
    if verbose:
        print(f" الصياغات: {all_queries}")

    unique_chunks: dict = {}
    for q in all_queries:
        for chunk in retrieve_chunks_raw(q, namespace):
            cid = chunk['id']
            if cid not in unique_chunks:
                unique_chunks[cid] = chunk
            else:
                unique_chunks[cid]['score'] = max(
                    unique_chunks[cid]['score'], chunk['score']
                )

    raw_chunks = sorted(unique_chunks.values(), key=lambda x: x['score'], reverse=True)
    if verbose:
        print(f" Chunks قبل reranking: {len(raw_chunks)}")

    reranked = rerank_chunks(question, raw_chunks, top_k=TOP_K_FINAL)
    if verbose and reranked:
        print(f" أفضل rerank score: {reranked[0]['rerank_score']:.3f}")

    if not reranked:
        return f'[{who}]: لا توجد معلومات كافية في قاعدة البيانات للإجابة على هذا السؤال.'

    full_context = build_context(reranked)

    response = client.chat.completions.create(
        model='gpt-4o',
        messages=[
            {'role': 'system', 'content': system_prompt},
            {'role': 'user',   'content': f'السياق:\n{full_context}\n\nالسؤال: {question}'}
        ],
        temperature=0,
        max_tokens=1500
    )

    return f"[{who}]: {response.choices[0].message.content}"

In [17]:
q1 = "كيف يجب على صاحب العمل إعلان لائحة تنظيم العمل للعمال؟ "

print(legal_rag(q1, verbose=True))

 الشخصية: محمد | Namespace: labor
 الصياغات: ['كيف يجب على صاحب العمل إعلان لائحة تنظيم العمل للعمال؟ ', 'ما هي الإجراءات التي يجب على صاحب العمل اتباعها لإبلاغ العمال بلائحة تنظيم العمل؟', 'كيف يمكن لصاحب العمل أن يقوم بإعلام الموظفين بقواعد تنظيم العمل؟', 'ما هي الخطوات المطلوبة من صاحب العمل لإعلان لائحة تنظيم العمل للموظفين؟', 'يجب على صاحب العمل أن يقوم بإعلان لائحة تنظيم العمل للعمال بطريقة واضحة وشفافة، وذلك من خلال نشرها في مكان بارز داخل مقر العمل، حيث يمكن لجميع العمال الاطلاع عليها بسهولة. كما يتعين عليه توفير نسخة من اللائحة لكل عامل عند بدء العمل، بالإضافة إلى تنظيم جلسات توعوية لشرح محتويات اللائحة وأهميتها. يجب أن تتضمن اللائحة جميع الحقوق والواجبات المتعلقة بالعمال، بما في ذلك ساعات العمل، والإجازات، والجزاءات، وأي سياسات أخرى ذات صلة، لضمان فهم العمال لحقوقهم والتزاماتهم بشكل كامل.']
 Chunks قبل reranking: 19
 أفضل rerank score: 0.725
[محمد]: وفقاً لنظام العمل السعودي، يجب على صاحب العمل إعلان لائحة تنظيم العمل للعمال بطريقة تضمن اطلاعهم عليها وفهمهم لمحتواها. يتضمن ذل

In [18]:
q1 = "متى يجب على المنشأة تحديث بياناتها في المنصة المعتمدة في حال تغيير عنوانها؟ "
print(legal_rag(q1, verbose=True))

 الشخصية: الفريق القانوني | Namespace: both
 الصياغات: ['متى يجب على المنشأة تحديث بياناتها في المنصة المعتمدة في حال تغيير عنوانها؟ ', 'ما هو الوقت المحدد الذي يتعين على المؤسسة فيه تعديل معلوماتها على المنصة الرسمية عند تغيير موقعها؟', 'متى يتوجب على الكيان تحديث تفاصيله في النظام المعتمد في حال حدوث تغيير في عنوانه؟', 'ما هي المدة الزمنية التي يجب على الشركة الالتزام بها لتحديث بياناتها على المنصة عند تغيير عنوانها؟', 'وفقًا لنظام العمل السعودي، يتعين على المنشأة تحديث بياناتها في المنصة المعتمدة خلال مدة لا تتجاوز 30 يومًا من تاريخ تغيير عنوانها. يُعتبر تحديث البيانات أمرًا ضروريًا لضمان دقة المعلومات المسجلة، ولتجنب أي تبعات قانونية قد تنجم عن عدم التحديث. يجب على المنشآت الالتزام بهذا الإجراء لضمان استمرارية خدماتها وتسهيل التواصل مع الجهات المعنية.']
 Chunks قبل reranking: 4
 أفضل rerank score: 1.150
[الفريق القانوني]: يجب على المنشأة تحديث بياناتها خلال 30 يومًا في حال تغيير عنوانها. [1]


In [19]:
q1 = "هل المؤجر مسؤول عن العيوب الخفية في العقار؟"
print(legal_rag(q1, verbose=True))

 الشخصية: سارة | Namespace: rent
 الصياغات: ['هل المؤجر مسؤول عن العيوب الخفية في العقار؟', 'هل يتحمل المالك المسؤولية عن العيوب المخفية في الممتلكات المؤجرة؟', 'هل يلتزم المؤجر بإصلاح العيوب غير المرئية في العقار؟', 'هل يمكن تحميل المؤجر تبعات العيوب الخفية الموجودة في العقار المؤجر؟', 'بموجب نظام الإيجار السعودي، يُعتبر المؤجر مسؤولاً عن العيوب الخفية التي تؤثر على استخدام المستأجر للعقار. يُلزم المؤجر بإبلاغ المستأجر عن أي عيوب معروفة قبل إبرام عقد الإيجار، كما يتحمل مسؤولية إصلاح العيوب الخفية التي تظهر خلال فترة الإيجار، والتي لم يكن بإمكان المستأجر اكتشافها عند استلام العقار. في حال عدم قيام المؤجر بإصلاح هذه العيوب، يحق للمستأجر المطالبة بتخفيض الإيجار أو إنهاء العقد، وفقاً لما يقتضيه النظام.']
 Chunks قبل reranking: 18
 أفضل rerank score: 0.288
[سارة]: بناءً على نظام الإيجار، لا توجد معلومات محددة في السياق المقدم حول مسؤولية المؤجر عن العيوب الخفية في العقار. ومع ذلك، عادةً ما يكون المؤجر مسؤولاً عن إصلاح العيوب الخفية التي تؤثر على سلامة المبنى أو تعيق انتفاع المستأجر، كما هو

In [20]:
q1 = " هل يحق لصاحب العمل تحديد موعد الإجازة؟"
print(legal_rag(q1, verbose=True))

 الشخصية: محمد | Namespace: labor
 الصياغات: [' هل يحق لصاحب العمل تحديد موعد الإجازة؟', 'هل يملك صاحب العمل السلطة لتحديد مواعيد الإجازات؟', 'هل يحق لصاحب العمل فرض مواعيد معينة للإجازة؟', 'هل يمكن لصاحب العمل التحكم في توقيت الإجازات للموظفين؟', 'وفقًا لنظام العمل السعودي، يحق لصاحب العمل تحديد مواعيد الإجازات وفقًا لاحتياجات العمل، مع مراعاة حقوق العمال المنصوص عليها في النظام. يجب على صاحب العمل إبلاغ الموظف بمواعيد الإجازة مسبقًا، ويجب أن تكون هذه المواعيد متوافقة مع الأنظمة واللوائح المعمول بها، بما في ذلك تحديد الحد الأدنى من الإجازات السنوية المستحقة للموظف. كما يحق للموظف تقديم طلب لتغيير موعد إجازته، ويجب على صاحب العمل النظر في هذا الطلب بجدية.']
 Chunks قبل reranking: 16
 أفضل rerank score: 2.955
[محمد]: وفقاً لنظام العمل السعودي، يحق لصاحب العمل تحديد مواعيد الإجازة السنوية وفقاً لظروف العمل، كما هو موضح في البند 8 من نموذج عقد العمل. يجب على صاحب العمل مراعاة تنظيم الإجازات بما يتوافق مع احتياجات العمل، ولكن يجب أيضاً ضمان حق العامل في الحصول على إجازته السنوية المستحقة.
